In [1]:
# ========Setup & Imports =========
import sys
from pathlib import Path
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name in ["notebook", "notebooks"] else Path.cwd()
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

from src.client import complete_gap


In [2]:
# ========Knowledge Base (Fake Facts) =========
docs = {
    "doc_gold": "The secret ingredient in the X-102 energy drink is 'Blue Moss Extract' harvested from the Caspian Sea.",
    "distractor_1": "The global production of lithium is expected to double by 2030 due to EV demand.",
    "distractor_2": "Mount Everest is officially 8,848.86 meters tall according to recent surveys.",
    "distractor_3": "The first decentralized digital currency was created in 2009 by an unknown person."
}

question = "What is the secret ingredient in the X-102 energy drink?"
# answer --> Blue Moss Extract


In [8]:
# ========Lost-in-the-Middle Scenarios =========
scenarios = {
    "Beginning": [docs["doc_gold"], docs["distractor_1"], docs["distractor_2"], docs["distractor_3"]],
    "Middle":    [docs["distractor_1"], docs["doc_gold"], docs["distractor_2"], docs["distractor_3"]],
    "End":       [docs["distractor_1"], docs["distractor_2"], docs["distractor_3"], docs["doc_gold"]]
}

def format_rag_prompt(context_list, query):
    context_text = "\n\n".join([f"Document {i+1}:\n{text}" for i, text in enumerate(context_list)])
    return f"""Use the provided documents to answer the question.
If the answer is not in the documents, say 'I don't know'.

Context:
{context_text}

Question: {query}
Answer:"""

results_rag = []

for position, context_list in scenarios.items():
    prompt = format_rag_prompt(context_list, question)
    res = complete_gap(prompt, temperature=0)
    ans = res.choices[0].message.content.strip()

    results_rag.append({
        "Position": position,
        "Response": ans,
        "Is_Correct": "Blue Moss Extract" in ans
    })

df_rag = pd.DataFrame(results_rag)
display(df_rag)
df_rag.to_csv(root_path/"report"/"ImgPlt"/"03_RAG.csv",index=False)

,Position,Response,Is_Correct
0,Beginning,The secret ingredient in the X-102 energy drin...,True
1,Middle,The secret ingredient in the X-102 energy drin...,True
2,End,The secret ingredient in the X-102 energy drin...,True


In [4]:
# ========Hallucination Test =========
unanswerable_question = "What is the capital of the Martian Colony established in 2045?"

prompt_hallucination = format_rag_prompt(scenarios["Middle"], unanswerable_question)
res_hallucination = complete_gap(prompt_hallucination, temperature=0)

print("--- Hallucination Test ---")
print(f"Question: {unanswerable_question}")
print(f"Model Response: {res_hallucination.choices[0].message.content.strip()}")


--- Hallucination Test ---
Question: What is the capital of the Martian Colony established in 2045?
Model Response: I don't know.


In [9]:
# ======== Final Report Summary =========
print("=== RAG Experiment Results ===")
for index, row in df_rag.iterrows():
    status = "✅ Success" if row['Is_Correct'] else "❌ Failed/Lost"
    print(f"Position: {row['Position']} -> {status}")

print("\n=== Hallucination Analysis ===")
hallucination_response = res_hallucination.choices[0].message.content.strip()
is_grounded = "don't know" in hallucination_response.lower() or "not" in hallucination_response.lower()

status_h = "✅ Grounded (Refused hallucination)" if is_grounded else "❌ Hallucinated (Fabricated info)"
print(f"Hallucination Test Result: {status_h}")
print(f"Response: {hallucination_response}")


=== RAG Experiment Results ===
Position: Beginning -> ✅ Success
Position: Middle -> ✅ Success
Position: End -> ✅ Success

=== Hallucination Analysis ===
Hallucination Test Result: ✅ Grounded (Refused hallucination)
Response: I don't know.
